# Estudio 06 · Paso d de clima · Ventanas de clima (d1)

Preregistro `STEP_D` (commit `b7ad38c`, agregado el 2026-11-06 después de ver b1, b2 y b3; aprobado por Rosa con tres
respuestas y el criterio (c), commit `658d0bd`, antes de correr).

**d1** = L + 5a + 14 columnas en memoria: lluvia CHIRPS v3, temperatura media y humedad relativa de ERA5-Land en la
semana del origen o y medias de o−1..o−4, o−5..o−8 y o−9..o−12; días con >= 20 mm (R20mm del ETCCDI) en o−3..o y
o−7..o−4 desde el bronze diario de Earth Engine. Sin Niño.

**Criterios de adopción** (frente a L + 5a y frente a b3): (a) gana la alerta (F1) sin «pierde» ni «compensa» en el
conteo; (b) gana el conteo (razón MAE) sin «pierde» ni «compensa» en la alerta; (c) gana por el Brier sin «pierde» ni
«compensa» en la alerta ni en el conteo. La base pasa a d1 solo si gana frente a L + 5a.

In [1]:
import json
import sys
import unittest

import numpy as np
import pandas as pd

from denguekay.config.paths import GOLD, RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import probabilistic_output as sp
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1_history as p1
import phase1b_prime as pb
import phase3_onsets as p3
import phase5_counts as p5
import phase6_other_families as p6
import phase_b_rain as pbr
import phase_b3_full_climate as b3
import phase_d_weather_windows as pdw
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "paso_d.json"
OUTPUT_DIR = p3.OUTPUT_DIR.parent / "paso_d"
SEED = pr.STEP_D_SHAP_SEED
device = xgboost_settings()
inp = p3.phase3_inputs()
chirps = pbr.read_chirps()
era5 = pdw.era5_weekly(inp["panel"])
daily = pdw.read_daily_ee()
weekly_daily = pdw.weekly_from_daily(daily)
origins = pdw.build_d1(inp["origenes"], chirps, era5, weekly_daily)
# the validated 5-week means, only for the consistency guard G35b (they are not d1 columns)
origins = pbr.add_rain(origins, pbr.rain_by_origin_week(chirps))
origins = pb.add_humidity(origins, pb.humidity_by_origin_week(pb.era5_land_panel(inp["panel"])))
origins = b3.add_temperature(origins, b3.temperature_by_origin_week(b3.era5_land_temperature(inp["panel"])))
columns_base, hp = p3._config()
columns = [*columns_base, *pr.STEP_D_COLUMNS]
print(columns)
device

['hist_lag_a', 'hist_lag_b', 'hist_media_4', 'hist_semanas_pos_4', 'semana_epi_seno', 'semana_epi_coseno', 'propia_semanas_desde_ultimo_caso', 'propia_pendiente_4', 'propia_log_casos_temporada_anterior', 'precip_total_mm_s0', 'precip_total_mm_media_s1_4', 'precip_total_mm_media_s5_8', 'precip_total_mm_media_s9_12', 'temp_media_s0', 'temp_media_media_s1_4', 'temp_media_media_s5_8', 'temp_media_media_s9_12', 'hum_rel_media_s0', 'hum_rel_media_media_s1_4', 'hum_rel_media_media_s5_8', 'hum_rel_media_media_s9_12', 'dias_precip_20mm_s0_3', 'dias_precip_20mm_s4_7']


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas (antes de mirar resultados)

In [2]:
gold = pd.read_csv(GOLD / "meteo_socio_epi_piura_2017_2025_h2_gold.csv", dtype={"ubigeo": str},
                   float_precision="round_trip")
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
saved_b3 = json.loads((STUDY / "docs" / "metricas" / "paso_b3.json").read_text(encoding="utf-8"))
b3_preds = pdw.read_b3_predictions()
b3_variant = {"predicciones": b3_preds,
              "evaluaciones": {s: p5.evaluate_mixture(p, inp["serie"]) for s, p in b3_preds.items()}}


def run_tests(names):
    suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(n) for n in names)
    result = unittest.TextTestRunner(verbosity=0).run(suite)
    return {"tests": names, "corridos": result.testsRun, "fallas": len(result.failures) + len(result.errors),
            "pasa": result.wasSuccessful()}


sys.path.insert(0, str(STUDY / "tests"))
synthetic = ["test_phase_d_weather_windows.TestWindows.test_window_definitions",
             "test_phase_d_weather_windows.TestWindows.test_heavy_rain_days"]
guards = {
    "G17": pb.guard_g17(gold, inp["panel"]),
    "G25": pbr.guard_definition(gold, inp["panel"]),
    "G29": p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"]),
    "G30": b3.guard_temperature_definition(inp["panel"]),
    "G35a": pdw.guard_daily_matches_silver(weekly_daily, chirps),
    "G35b": pdw.guard_consistency(origins),
    "G35c": run_tests(synthetic),
    "G36_sintetica": run_tests(["test_phase_d_weather_windows.TestWindows.test_no_leakage_by_perturbation"]),
    "G36_real": pdw.guard_no_leakage_real(inp["origenes"], chirps, era5, weekly_daily, pr.STEP_D_NO_LEAK_CUT),
    "G37": pdw.guard_coverage(origins),
    "G38": pdw.guard_availability(origins),
    "G39": p6.guard_g16_reproduction(b3_variant["evaluaciones"], saved_b3["veredicto_alerta"]["metricas"]),
}
failed = [k for k, g in guards.items() if not g["pasa"]]
if failed:
    raise RuntimeError(f"Fallan guardas {failed}: se detiene el paso d. {guards}")
guards

----------------------------------------------------------------------
Ran 2 tests in 0.225s

OK


----------------------------------------------------------------------
Ran 1 test in 0.157s

OK


{'G17': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G25': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G29': {'diferencia_maxima': 4.9451664985156185e-11,
  'filas': 18,
  'pasa': True},
 'G30': {'filas': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G35a': {'semanas_silver': 90675,
  'semanas_solo_en_silver': 0,
  'semanas_solo_en_diario': 0,
  'mismos_vacios': True,
  'diferencia_maxima_mm': 0.0,
  'pasa': True},
 'G35b': {'por_fuente': {'precip_total_mm': {'columna_referencia': 'precip_total_mm_media5_h2',
    'mismos_vacios': True,
    'diferencia_maxima': 5.684341886080802e-14,
    'pasa': True},
   'temp_media': {'columna_referencia': 'temp_media_media5_h2',
    'mismos_vacios': True,
    'diferencia_maxima': 7.105427357601002e-15,
    'pasa': True},
   'hum_rel_media': {'columna_refer

## 2 · Variante d1 (5 semillas)

In [3]:
blocks = p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
preds, evals, fits = {}, {}, []
for seed in pr.SEEDS:
    pred = p3.predict_blocks_weighted(inp["matrices"], origins, blocks, seed, columns, hp)
    fits.append(pred.attrs.pop("ajustes").assign(semilla=seed))
    preds[seed] = pred
    evals[seed] = p5.evaluate_mixture(pred, inp["serie"])
cand = {"predicciones": preds, "evaluaciones": evals}
pd.concat([p.assign(semilla=s) for s, p in preds.items()], ignore_index=True).to_csv(
    OUTPUT_DIR / "predicciones_d1.csv.gz", index=False)
pd.concat(fits, ignore_index=True).drop(columns=["corte"]).to_csv(OUTPUT_DIR / "ajustes_d1.csv", index=False)

## 3 · Veredictos preregistrados (criterios a, b y c, frente a L + 5a y frente a b3)

In [4]:
results = {}
for ref_name, ref in (("L + 5a", base), ("b3", b3_variant)):
    alert, count, brier = p3.compare(cand, ref), p5.compare(cand, ref), pdw.brier_compare(cand, ref)
    crit = pr.adopt_step_d_criteria(alert["veredicto"], count["veredicto"], brier["veredicto"])
    results[ref_name] = {"alerta": alert, "conteo": count, "brier": brier, "criterios": crit}
    print(f"d1 frente a {ref_name}:", "a", alert["veredicto"].veredicto.tolist(), "| b", count["veredicto"].veredicto.tolist(),
          "| c", brier["veredicto"].veredicto.tolist(), "|", crit)
adopted = results["L + 5a"]["criterios"]["adopta"]
loses_to_b3 = any(results["b3"][k]["veredicto"].veredicto.eq("pierde").any() for k in ("alerta", "conteo", "brier"))
print("Base resultante:", "L + 5a + d1" if adopted else "L + 5a", "| d1 pierde en algo frente a b3:", loses_to_b3)

d1 frente a L + 5a: a ['empate', 'empate', 'empate'] | b ['empate', 'empate', 'empate'] | c ['empate', 'empate', 'empate'] | {'a_alerta': False, 'b_conteo': False, 'c_brier': False, 'adopta': False}


d1 frente a b3: a ['empate', 'empate', 'empate'] | b ['empate', 'empate', 'empate'] | c ['empate', 'empate', 'empate'] | {'a_alerta': False, 'b_conteo': False, 'c_brier': False, 'adopta': False}
Base resultante: L + 5a | d1 pierde en algo frente a b3: False


In [5]:
for ref_name, r in results.items():
    for k in ("alerta", "conteo", "brier"):
        print(f"--- d1 frente a {ref_name} · {k}")
        display(r[k]["veredicto"])
display(results["L + 5a"]["alerta"]["antelacion"].round(3))
results["b3"]["alerta"]["antelacion"].round(3)

--- d1 frente a L + 5a · alerta


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,auprc,razon_mae,1,1,2,3,1,3,0,1,2,1
1,3,empate,False,False,NaN,razon_mae,0,1,2,3,2,2,0,1,0,1
2,4,empate,False,False,NaN,"brier, razon_mae",2,2,3,3,2,2,1,1,0,1


--- d1 frente a L + 5a · conteo


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,2,empate,auprc,razon_rmse,3,3,1,1,2,1,1,1,3,0,2,1
1,3,empate,NaN,razon_rmse,3,3,0,1,2,1,1,2,2,0,0,1
2,4,empate,NaN,"razon_rmse, brier",3,3,2,2,3,1,1,2,2,1,0,1


--- d1 frente a L + 5a · brier


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_brier,a_favor_auprc,a_favor_f1_050,a_favor_razon_mae,en_contra_brier,en_contra_auprc,en_contra_f1_050,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,auprc,razon_mae,2,1,1,3,0,3,1,1,2,1
1,3,empate,NaN,razon_mae,2,1,0,3,0,2,2,1,0,1
2,4,empate,NaN,razon_mae,3,2,2,3,1,2,2,1,0,1


... (4 outputs left out)

--- d1 frente a b3 · brier


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_brier,a_favor_auprc,a_favor_f1_050,a_favor_razon_mae,en_contra_brier,en_contra_auprc,en_contra_f1_050,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,NaN,None,0,2,1,0,2,2,0,2,2,2
1,3,empate,auprc,None,0,2,0,0,3,3,1,2,2,0
2,4,empate,auprc,None,1,1,0,2,3,3,2,2,1,0


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.021,-0.016,0.057,0.067,-0.029,0.118,0.026,-0.035,30,34,34,38,57
1,3,193,-0.016,-0.052,0.016,0.000,0.000,0.029,0.000,-0.070,30,34,34,38,57
2,4,193,-0.010,-0.047,0.021,-0.033,0.000,0.029,0.000,-0.035,30,34,34,38,57


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.021,-0.010,0.057,0.000,-0.059,0.147,0.079,-0.035,30,34,34,38,57
1,3,193,0.010,-0.016,0.036,0.000,-0.029,0.059,0.053,-0.018,30,34,34,38,57
2,4,193,0.005,-0.026,0.036,-0.033,0.000,0.029,0.053,-0.018,30,34,34,38,57


## 4 · Métricas reportadas (no deciden): Brier, BSS frente a la climatología y AUPRC general y bajo el umbral

In [6]:
report_blocks = [pr.block_name(s) for s in pr.STEP_D_REPORT_SEASONS]
variants = {"L + 5a": base, "b3": b3_variant, "d1": cand}
reported = pd.concat({k: pdw.reported_by_variant(v, inp["serie"], report_blocks) for k, v in variants.items()},
                     names=["variante"]).reset_index(level=0)
show = ["variante", "horizonte", "bloque", "positivos", "brier", "brier_climatologia", "bss_climatologia", "auprc",
        "n_bajo_umbral", "positivos_bajo_umbral", "auprc_bajo_umbral"]
reported[show].sort_values(["bloque", "horizonte", "variante"]).round(3)

,variante,horizonte,bloque,positivos,brier,brier_climatologia,bss_climatologia,auprc,n_bajo_umbral,positivos_bajo_umbral,auprc_bajo_umbral
0,L + 5a,2,temporada_2010,234.0,0.030,0.063,0.523,0.780,365.0,24.0,0.389
0,b3,2,temporada_2010,234.0,0.030,0.063,0.528,0.789,365.0,24.0,0.423
0,d1,2,temporada_2010,234.0,0.030,0.063,0.524,0.779,365.0,24.0,0.379
6,L + 5a,3,temporada_2010,234.0,0.036,0.063,0.432,0.706,356.0,29.0,0.345
6,b3,3,temporada_2010,234.0,0.035,0.063,0.444,0.724,356.0,29.0,0.375
6,d1,3,temporada_2010,234.0,0.036,0.063,0.437,0.702,356.0,29.0,0.397
12,L + 5a,4,temporada_2010,234.0,0.042,0.063,0.339,0.600,362.0,46.0,0.450
12,b3,4,temporada_2010,234.0,0.041,0.063,0.357,0.632,362.0,46.0,0.489
12,d1,4,temporada_2010,234.0,0.041,0.063,0.348,0.612,362.0,46.0,0.497
1,L + 5a,2,temporada_2015,557.0,0.051,0.140,0.635,0.885,322.0,51.0,0.454


In [7]:
# Comprobación: el Brier y la AUPRC de reported_metrics coinciden con las del protocolo (mediana de semillas)
med = {k: pa.seed_median([e["metricas"] for e in v["evaluaciones"].values()]) for k, v in variants.items()}
check = reported.merge(pd.concat({k: m for k, m in med.items()}, names=["variante"]).reset_index(level=0),
                       on=["variante", "horizonte", "bloque"], suffixes=("", "_protocolo"))
check_diff = {"filas": len(check), "brier": float((check.brier - check.brier_protocolo).abs().max()),
              "auprc": float((check.auprc - check.auprc_protocolo).abs().max())}
check_diff

{'filas': 54, 'brier': 0.0, 'auprc': 0.0}

## 5 · Lecturas que no deciden

In [8]:
# Criterio (c) para b3 frente a L + 5a: declarado como posterior a ver los resultados de b3
b3_alert, b3_count = p3.compare(b3_variant, base), p5.compare(b3_variant, base)
b3_brier = pdw.brier_compare(b3_variant, base)
b3_crit = pr.adopt_step_d_criteria(b3_alert["veredicto"], b3_count["veredicto"], b3_brier["veredicto"])
print(b3_crit)
b3_brier["veredicto"]

{'a_alerta': False, 'b_conteo': False, 'c_brier': True, 'adopta': True}


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_brier,a_favor_auprc,a_favor_f1_050,a_favor_razon_mae,en_contra_brier,en_contra_auprc,en_contra_f1_050,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,gana,None,razon_mae,4,2,1,3,0,2,1,1,1,1
1,3,gana,None,"auprc, razon_mae",4,3,1,3,0,2,1,1,0,2
2,4,empate,None,"auprc, razon_mae",3,3,2,3,0,2,1,1,0,1


In [9]:
# HU0007-4 por celda (2022 y 2023)
hu = pd.concat({k: sp.hu0007_4_compliance(m[m.bloque.isin(["temporada_2022", "temporada_2023"])]) for k, m in med.items()},
               names=["variante"]).reset_index(level=0)
hu[["variante", "horizonte", "bloque", "cumple_clasificacion", "cumple_regresion"]]

,variante,horizonte,bloque,cumple_clasificacion,cumple_regresion
15,L + 5a,2,temporada_2022,False,False
16,L + 5a,2,temporada_2023,True,False
33,L + 5a,3,temporada_2022,False,False
34,L + 5a,3,temporada_2023,False,False
51,L + 5a,4,temporada_2022,False,False
52,L + 5a,4,temporada_2023,False,True
15,b3,2,temporada_2022,False,False
16,b3,2,temporada_2023,True,True
33,b3,3,temporada_2022,False,True
34,b3,3,temporada_2023,False,True


In [10]:
# SHAP de d1 por horizonte (semilla 17)
shap = {b: b3.shap_by_horizon(b3.fit_block(inp, origins, b, SEED, columns, hp)) for b in pr.STEP_D_SHAP_BLOCKS}
pd.concat(shap, names=["bloque", "horizonte"]).round(4).T

bloque                              temporada_2022                 temporada_2023                 temporada_2024                
horizonte                                      h=2     h=3     h=4            h=2     h=3     h=4            h=2     h=3     h=4
hist_lag_a                                  0.0550  0.0915  0.1229         0.0784  0.1214  0.1571         0.0864  0.1373  0.1809
hist_lag_b                                  0.0230  0.0168  0.0167         0.0119  0.0098  0.0150         0.0082  0.0068  0.0168
hist_media_4                                0.2603  0.2225  0.1849         0.2863  0.2481  0.2074         0.2355  0.1991  0.1695
hist_semanas_pos_4                          0.0000  0.0000  0.0000         0.0000  0.0000  0.0000         0.0066  0.0056  0.0043
semana_epi_seno                             0.1280  0.1613  0.1808         0.1447  0.1715  0.1882         0.1295  0.1575  0.1790
semana_epi_coseno                           0.0555  0.0912  0.1313         0.0714  0.1144  0.1606         0.0435  0.0781  0.1135
propia_semanas_desde_ultimo_caso            1.5334  1.4218  1.3102         1.4888  1.3756  1.2747         1.3652  1.2526  1.1499
propia_pendiente_4                          0.0018  0.0038  0.0055         0.0017  0.0021  0.0021         0.0014  0.0015  0.0011
propia_log_casos_temporada_anterior         0.0321  0.0393  0.0434         0.0372  0.0469  0.0539         0.0645  0.0751  0.0852
precip_total_mm_s0                          0.0158  0.0255  0.0318         0.0154  0.0244  0.0299         0.0106  0.0190  0.0262
precip_total_mm_media_s1_4                  0.0376  0.0426  0.0412         0.0222  0.0280  0.0286         0.0685  0.0820  0.0907
precip_total_mm_media_s5_8                  0.0159  0.0209  0.0218         0.0167  0.0214  0.0230         0.0300  0.0347  0.0339
precip_total_mm_media_s9_12                 0.0042  0.0061  0.0074         0.0022  0.0034  0.0042         0.0098  0.0197  0.0298
temp_media_s0                               0.1902  0.2152  0.2382         0.1917  0.2131  0.2346         0.1984  0.2208  0.2390
temp_media_media_s1_4                       0.1144  0.1527  0.1850         0.0944  0.1295  0.1593         0.0874  0.1170  0.1397
temp_media_media_s5_8                       0.0413  0.0521  0.0630         0.0393  0.0512  0.0618         0.0337  0.0428  0.0494
temp_media_media_s9_12                      0.0125  0.0139  0.0148         0.0160  0.0187  0.0209         0.0044  0.0059  0.0078
hum_rel_media_s0                            0.0174  0.0248  0.0331         0.0136  0.0181  0.0229         0.0102  0.0171  0.0246
hum_rel_media_media_s1_4                    0.0151  0.0210  0.0232         0.0133  0.0170  0.0198         0.0269  0.0326  0.0339
hum_rel_media_media_s5_8                    0.0128  0.0154  0.0160         0.0100  0.0135  0.0134         0.0070  0.0090  0.0090
hum_rel_media_media_s9_12                   0.0309  0.0398  0.0529         0.0183  0.0242  0.0318         0.0662  0.0780  0.0924
dias_precip_20mm_s0_3                       0.0219  0.0265  0.0293         0.0144  0.0169  0.0187         0.0580  0.0730  0.0828
dias_precip_20mm_s4_7                       0.0043  0.0048  0.0046         0.0019  0.0021  0.0018         0.0004  0.0003  0.0002

## 6 · Salidas: métricas y MLflow

In [11]:
to_json = lambda d: {k: diag.to_json(x) for k, x in d.items() if isinstance(x, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": guards,
    "comparaciones": {ref: {"alerta": to_json(r["alerta"]), "conteo": to_json(r["conteo"]), "brier": to_json(r["brier"]),
                            "criterios": r["criterios"]} for ref, r in results.items()},
    "adoptada": bool(adopted), "base_resultante": "L + 5a + d1" if adopted else "L + 5a",
    "pierde_frente_a_b3": bool(loses_to_b3),
    "metricas_reportadas": diag.to_json(reported), "comprobacion_metricas_reportadas": check_diff,
    "criterio_c_b3_frente_a_L_5a": {"criterios": b3_crit, "veredicto": diag.to_json(b3_brier["veredicto"])},
    "hu0007_4_2022_2023": diag.to_json(hu),
    "shap_por_horizonte_semilla_17": {b: diag.to_json(t.reset_index(names="horizonte")) for b, t in shap.items()},
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdicts = pd.concat({f"{ref} · {k}": r[k]["veredicto"][["horizonte", "veredicto"]] for ref, r in results.items()
                      for k in ("alerta", "conteo", "brier")}, names=["regla"]).reset_index(level=0)
run_id = tracking.log_study(
    "mejora_modelo_paso_b_lluvia_chirps", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "paso_d", "preregistro": "b7ad38c + 658d0bd", "adoptada_d1": str(bool(adopted)),
                       "base_resultante": content["base_resultante"],
                       "veredicto": "; ".join(f"{r.regla} h{r.horizonte}: {r.veredicto}" for r in verdicts.itertuples()),
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto": verdicts},
    description="Paso d del estudio 06: ventanas de clima (lluvia CHIRPS, temperatura y humedad ERA5-Land, días >= 20 mm) sobre L + 5a.")
run_id

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 851bf57f64214ed48d38c84914304f71 en 'denguekay/mejora_modelo_paso_b_lluvia_chirps' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'851bf57f64214ed48d38c84914304f71'